# Van der Waal's calculator
This notebook uses the Van der Waal's equation of state (VdW-EoS) to calculate the volume of fluid, provided the temperature, pressure, and critical properties. The VdW-EoS is given below:

$$ P = \frac{RT}{V-b} - \frac{a}{V^2} $$

Where:

$$ a = \frac{27 (R T_C)^2}{64 P_C} $$
$$ b = \frac{R T_c}{8 P_C} $$

The notebook uses a non-linear root finding algorithm provided by `scipy.optimize.fsolve` to determine the volume. Whether the converged volume is the liquid- or vapour-phase volume therefore depends on the initial guess.

## 1. Import libraries and define inputs

Simply uncomment the critical properties of the component of interest. Make sure the properties for all other components are commented out.

You can easily comment / uncomment by highlighting the relevant lines of code and pressing `CTRL+/`.

In [1]:
import numpy as np 
from scipy.optimize import fsolve #to solve vdW cubic equation of state

R = 8.314462618 #defining the universal gas constant [J/(mol.K)]

# Critical properties: H2
T_c = 33.145 # critical temperature  [K]
P_c = 1.2964 # critical pressure [MPa]
P_c = P_c*1e6 # converting critical pressure from MPa to Pa

# Critical properties: CO2
# T_c = 304.13 # critical temperature  [K]
# P_c = 7.38 # critical pressure [MPa]
# P_c = P_c*1e6 # converting critical pressure from MPa to Pa

## 2. Calculating the VdW coefficients $a$ and $b$

In [2]:
a = 27/64*((R**2)*(T_c**2)/P_c) #calculating parameter for intermolecular attractive forces [Pa.m^6/(m^2)]
print("The value for parameter a is: ", a)
b = 1/8*R*T_c/P_c # parameter for finite molecular volume
print("The value for parameter b is: ", b) #check

The value for parameter a is:  0.024714349536317987
The value for parameter b is:  2.6571936080068846e-05


## 3. Solving the cubic EoS

In [3]:
T = 35 # Temperature [K]
P = 1.3 # Pressure [MPa]
P = P*1e6 #converting pressure from MPa to Pa
V0_v = R*T/P #initial guess for vapour volume root [m^3/mol]
V0_l = b * 1.1 #initial guess for liquid volume root [m^3/mol]
def func (V):
    return (P - (R*T/(V-b)) + (a/(V**2)))

V_v = fsolve(func, (V0_v)) #solving the cubic equation of state for vapour volume
V_l = fsolve(func, (V0_l)) #solving the cubic equation of state for liquid volume

print("The possible value for system vapour volume is: ", V_v, "m^3/mol") #point check
print("The possible value for system liquid volume is: ", V_l, "m^3/mol") #point check

The possible value for system vapour volume is:  [0.000141] m^3/mol
The possible value for system liquid volume is:  [0.000141] m^3/mol


## 4. Finalising the vapor and liquid roots

In [4]:
tol = 1e-6 
vapour_exists = abs(func(V_v[0])) < tol * P #check if the vapour solution is close to zero
liquid_exists = abs(func(V_l[0])) < tol * P #check if the liquid solution is close to zero
same_root = abs (V_v[0] - V_l[0]) < tol * V_l[0] #check if the two roots are the same

sc = T > T_c and P > P_c #check if the system is in a supercritical state

if sc:
    V_sc = V_l[0] if liquid_exists else V_v[0] #if the system is supercritical, assign the volume to the liquid root if it exists, otherwise assign it to the vapour root
    print("The system is in a supercritical state. The volume is: ", V_sc, "m^3/mol")

elif vapour_exists and liquid_exists and not same_root:
    print("The system has two distinct roots. The vapour volume is: ", V_v, "m^3/mol and the liquid volume is: ", V_l, "m^3/mol")

elif vapour_exists or liquid_exists:
    V_single = V_v[0] if vapour_exists else V_l[0] #if only one root exists, assign the volume to the existing root
    if V_single < 3*b:
        print("The system has one liquid-like root. The volume is: ", V_single, "m^3/mol")
    else:
        print("The system has one vapour-like root. The volume is: ", V_single, "m^3/mol")

else:
    print("No valid roots found for the given temperature and pressure.")

The system is in a supercritical state. The volume is:  0.00014100490955858174 m^3/mol
